# Validación automática del conjunto candidato

Este notebook ejecuta las siete comprobaciones de calidad exigidas sobre el CSV candidato. Todas las reglas provienen de `src/validadores.py`, la misma implementación utilizada por `pytest`, y cada fallo incluye cantidad, explicación y ejemplos suficientes para corregirlo.

## Configuraci?n y carga reproducible

In [7]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la ra?z del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src.validadores import (
    cargar_csv_para_validacion,
    ejecutar_validaciones,
    resumen_validaciones,
)

In [8]:
RUTA_CANDIDATO = ROOT / "data" / "processed" / "establecimientos_limpios_candidato.csv"
df_candidato = cargar_csv_para_validacion(RUTA_CANDIDATO)

print(f"Registros: {df_candidato.shape[0]:,}")
print(f"Variables: {df_candidato.shape[1]}")
df_candidato.head()

Registros: 11,868
Variables: 18


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ZONA_CAPITAL,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,<NA>,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


## Ejecución de las siete validaciones

In [9]:
resultados = ejecutar_validaciones(df_candidato, max_ejemplos=5)
resumen = resumen_validaciones(resultados)
resumen

,Prueba,Estado,Errores,Detalle
0,Duplicados exactos,APROBADO,0,No deben existir filas idénticas en todas las ...
1,Espacios en textos,APROBADO,0,"Los textos no deben tener espacios al inicio, ..."
2,Formato de teléfonos,APROBADO,0,"Cada teléfono debe tener 8 dígitos, iniciar en..."
3,Geografía oficial,APROBADO,0,Departamento y municipio deben pertenecer al c...
4,Esquema y tipos,APROBADO,0,El candidato debe contener las 18 columnas esp...
5,Categorías equivalentes,APROBADO,0,No deben coexistir categorías que solo difiera...
6,Valores inválidos diagnosticados,APROBADO,0,"Códigos, distritos, dominios, faltantes obliga..."


Las pruebas comprueban, en orden: duplicados exactos; espacios en textos; teléfonos; geografía oficial; esquema y tipos; categorías equivalentes por escritura; y valores inválidos identificados durante el diagnóstico.

## Detalle de fallos y ejemplos

In [10]:
fallos = [resultado for resultado in resultados if not resultado.aprobada]
if not fallos:
    print("No se encontraron fallos; las siete validaciones fueron aprobadas.")
else:
    for resultado in fallos:
        print(f"\n{resultado.prueba}: {resultado.errores} error(es)")
        print(resultado.detalle)
        print(pd.DataFrame(resultado.ejemplos).to_string(index=False))

No se encontraron fallos; las siete validaciones fueron aprobadas.


## Comprobación inequívoca del resultado

In [11]:
assert len(resultados) == 7, "Deben ejecutarse exactamente siete validaciones."
assert all(resultado.aprobada for resultado in resultados), {
    resultado.prueba: resultado.ejemplos for resultado in fallos
}
assert resumen["Errores"].sum() == 0

print("VALIDACIÓN APROBADA: 7 de 7 pruebas superadas con cero errores.")

VALIDACIÓN APROBADA: 7 de 7 pruebas superadas con cero errores.


## Esquema validado

In [12]:
pd.DataFrame(
    {
        "Variable": df_candidato.columns,
        "Tipo observado": [str(tipo) for tipo in df_candidato.dtypes],
        "Valores faltantes": [int(df_candidato[col].isna().sum()) for col in df_candidato],
        "Valores ?nicos": [int(df_candidato[col].nunique(dropna=True)) for col in df_candidato],
    }
)

,Variable,Tipo observado,Valores faltantes,Valores ?nicos
0,CODIGO,string,0,11868
1,DISTRITO,string,603,1678
2,DEPARTAMENTO,string,0,22
3,MUNICIPIO,string,0,330
4,ZONA_CAPITAL,string,9707,22
5,ESTABLECIMIENTO,string,5,6312
6,DIRECCION,string,89,7431
7,TELEFONO,string,1063,6459
8,SUPERVISOR,string,539,1279
9,DIRECTOR,string,2142,5489


## Resultado

El conjunto candidato supera las siete reglas automáticas y puede avanzar a la comparación formal Antes/Después. Este resultado no renombra ni exporta todavía el CSV final.